# Práctica 1 - Lingüística Computacional


# Fonética

In [1]:
from collections import defaultdict

import pandas as pd
import requests

In [2]:
IPA_URL = "https://raw.githubusercontent.com/open-dict-data/ipa-dict/master/data/{lang}.txt"

In [3]:
response = requests.get(IPA_URL.format(lang="es_MX"))

In [4]:
ipa_list = response.text.split("\n")

In [5]:
ipa_list[0].split("\t")

['a', '/a/']

In [6]:
def download_ipa_corpus(iso_lang: str) -> str:
    """
    Descarga el archivo del diccionario IPA para el idioma dado.
    """
    print(f"Descargando {iso_lang}...", end=" ")
    response = requests.get(IPA_URL.format(lang=iso_lang))
    print(f"status={response.status_code}")
    
    if response.status_code != 200:
        print(f"Error al descargar el corpus para {iso_lang}")
        return ""
    
    return response.text

In [7]:
def parse_response(response: str) -> dict:
    """
    Convierte el texto crudo del diccionario IPA en un diccionario de Python.
    Formato esperado por línea: palabra[TAB]ipa
    """
    ipa_list = response.rstrip().split("\n")
    result = {}
    
    for item in ipa_list:
        if item == "":
            continue
        
        item_list = item.split("\t")
        
        if len(item_list) == 2:
            word, ipa = item_list
            result[word] = ipa
    
    return result

In [8]:
es_data = parse_response(download_ipa_corpus("es_MX"))

Descargando es_MX... status=200


In [9]:
def get_ipa_transcriptions(word: str, dataset: dict) -> list[str]:
    """
    Busca una palabra en el diccionario y devuelve sus transcripciones IPA.
    Si no existe, devuelve lista vacía.
    """
    return dataset.get(word.lower(), "").split(", ") if dataset.get(word.lower(), "") else []

In [10]:
get_ipa_transcriptions("mayonesa", es_data)

['/maʝonesa/']

In [11]:
def levenshtein(s1: str, s2: str) -> int:
    """
    Calcula la distancia de Levenshtein entre dos cadenas.
    """
    m = len(s1)
    n = len(s2)

    dp = [[0 for _ in range(n + 1)] for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                costo = 0
            else:
                costo = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + costo
            )

    return dp[m][n]

In [12]:
def encontrar_palabra_mas_cercana(palabra: str, dataset: dict) -> str:
    """
    Encuentra la palabra más cercana en el diccionario usando distancia de Levenshtein.
    """

    palabra_mas_cercana = None
    distancia_minima = float("inf")

    for palabra_dic in dataset.keys():
        distancia = levenshtein(palabra.lower(), palabra_dic.lower())

        if distancia < distancia_minima:
            distancia_minima = distancia
            palabra_mas_cercana = palabra_dic

    return palabra_mas_cercana

In [13]:
def obtener_ipa_aproximado(palabra: str, dataset: dict) -> dict:
    """
    Busca la transcripción IPA de una palabra.
    Si no existe en el diccionario, aproxima usando la palabra más cercana.
    """

    resultado_exacto = get_ipa_transcriptions(palabra, dataset)

    if resultado_exacto:
        return {
            "encontrada": True,
            "palabra_ingresada": palabra,
            "palabra_coincidente": palabra.lower(),
            "transcripciones": resultado_exacto
        }

    palabra_cercana = encontrar_palabra_mas_cercana(palabra, dataset)
    transcripcion_aprox = get_ipa_transcriptions(palabra_cercana, dataset)

    return {
        "encontrada": False,
        "palabra_ingresada": palabra,
        "palabra_coincidente": palabra_cercana,
        "transcripciones": transcripcion_aprox
    }

In [16]:
from pprint import pprint

In [17]:
print(obtener_ipa_aproximado("tomate", es_data))
print(obtener_ipa_aproximado("cassa", es_data))
print(obtener_ipa_aproximado("cucharacha", es_data))

{'encontrada': True, 'palabra_ingresada': 'tomate', 'palabra_coincidente': 'tomate', 'transcripciones': ['/tomate/']}
{'encontrada': False, 'palabra_ingresada': 'cassa', 'palabra_coincidente': 'cansa', 'transcripciones': ['/kansa/']}
{'encontrada': False, 'palabra_ingresada': 'cucharacha', 'palabra_coincidente': 'cucaracha', 'transcripciones': ['/kukaɾatʃa/']}


# Morfología